# Exploratory Data Analysis and Preprocessing

This notebook performs exploratory data analysis (EDA) and data preprocessing on the AI4I 2020 predictive maintenance dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries imported successfully')

## 1. Load Raw Data

In [ ]:
# Load raw dataset
df_raw = pd.read_csv('../data/raw/ai4i2020.csv')

print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumn names:\n{df_raw.columns.tolist()}")
print(f"\nFirst few rows:")
df_raw.head()

## 2. Data Overview

In [ ]:
# Basic information
print("Data Info:")
df_raw.info()

print("\nDescriptive Statistics:")
df_raw.describe()

## 3. Missing Values Analysis

In [ ]:
# Check for missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

print("Missing Values:")
print(pd.DataFrame({'Count': missing, 'Percentage': missing_pct}))

## 4. Target Variable Distribution

In [ ]:
# Analyze machine failure distribution
if 'Machine failure' in df_raw.columns:
    failure_counts = df_raw['Machine failure'].value_counts()
    failure_pct = (failure_counts / len(df_raw) * 100).round(2)
    
    print("Machine Failure Distribution:")
    print(failure_counts)
    print(f"\nPercentages:")
    print(failure_pct)
    
    # Plot
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df_raw, x='Machine failure')
    plt.title('Machine Failure Distribution')
    plt.show()

## 5. Sensor Data Distribution

In [ ]:
# Sensor columns
sensor_cols = ['Air temperature [K]', 'Process temperature [K]', 
               'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(sensor_cols):
    if col in df_raw.columns:
        sns.histplot(df_raw[col], kde=True, ax=axes[idx])
        axes[idx].set_title(f'Distribution of {col}')

# Hide empty subplot
axes[5].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Correlation matrix
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns
correlation_matrix = df_raw[numeric_cols].corr()

# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

## 7. Run Data Cleaning Pipeline

In [ ]:
import sys
sys.path.append('../src')

from data_cleaning import DataCleaner

# Initialize cleaner
cleaner = DataCleaner(outlier_threshold=3.0)

# Run full pipeline
df_cleaned = cleaner.full_pipeline(
    filepath='../data/raw/ai4i2020.csv',
    handle_imbalance=True,
    outlier_method='iqr'
)

print(f"Cleaned dataset shape: {df_cleaned.shape}")

## 8. Save Cleaned Data

In [ ]:
# Save cleaned data
df_cleaned.to_csv('../data/processed/ai4i2020_cleaned.csv', index=False)
print("Cleaned data saved to ../data/processed/ai4i2020_cleaned.csv")

## Summary

This notebook performed:
- Data loading and exploration
- Missing value analysis
- Target variable distribution analysis
- Sensor data visualization
- Correlation analysis
- Data cleaning pipeline execution
- Cleaned data export